In [1]:
import requests
import pandas as pd
import re
import urllib
from tqdm import tqdm

# This notebook converts voting_id's to dataframes of votes for each period

In [ ]:
with urllib.request.urlopen("https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/folketinget_votings_enriched.csv") as response: #UPDATE TO MATCH THE NEW NAME THIS IS AN OLD FILE
    df_voting_sessions = pd.read_csv(response)

In [3]:
n_votes = len(df_voting_sessions['id_afstemning'].unique())
n_periods = df_voting_sessions['Period'].nunique()
print(f"There are {n_votes} unique voting sessions in the dataset, across {n_periods} periods.")

There are 10304 unique voting sessions in the dataset, across 7 periods.


In [4]:
# base_url = "https://oda.ft.dk/api/"
    
# n_votes = 500
# def get_voting_sessions_in_period():
#     # Get all voting ID´s in a specific period to pass to "get_voting_session_with_votes"
#     # voting_ids= [10375, 10376, 10377, 10378] #
#     voting_ids = range(10379-n_votes, 10379)
#     return voting_ids

def get_voting_sessions_in_period(df_voting_sessions, period):
    vote_ids = df_voting_sessions[df_voting_sessions['Period'] == period]['id_afstemning'].unique() #Do i need to convert to list?
    print(f"Found {len(vote_ids)} votes in period {period}")
    return vote_ids

# def get_data_from_voting_session(voting_id):
#     data = DanishParliamentAPI().get_voting_session_details(voting_id)
#     vote_data = []
#     votes = data.get('Stemme')
#     for individual_vote in votes:
#         # return_data = individual_vote
#         # return_data = individual_vote.get('Aktør').get('biografi') #This is useful for finding the information about the person who voted.

#         politician_bio = individual_vote.get('Aktør').get('biografi')
#         politician_party = re.search('<party>([^<]+)</party>', politician_bio).group(1) #Extract the party
#         politician_name = individual_vote.get('Aktør').get('navn') #Find the name of who votes. We will use this for naming the nodes.
#         vote_type = individual_vote.get('typeid') #What did they vote?

#         vote_data.append((voting_id, politician_party, politician_name, vote_type)) #Will pass to DF instead, so no need for dict 
#     df = pd.DataFrame(vote_data, columns = ['voting_id', 'party', 'politician', 'vote_type'])
#     return df
        # print(vote_data)
    

################################################## All is out-commented, to use the DanishParliamentAPI instead.
# BASE_URL = "https://oda.ft.dk/api/"
request_session = requests.Session()

def get_voting_session_with_votes(vote_id, session = request_session):
    base_url = "https://oda.ft.dk/api/"
    all_votes = []
    skip = 0

    next_link = None

    while True:
        # params = {
        #     '$top':100
        #     ,'$skip' : skip
        #     # ,'$filter': f'id eq {voting_id}'
        #     ,'$expand': 'Stemme' #/Aktør
        # }

        url = f"{base_url}Afstemning({vote_id})/Stemme"
        # print(f"Base URL without params: {url}")
        if next_link:
            # print(f"THERE IS A NEXT LINK: {next_link}")
            response = session.get(next_link)
        else:
            # response = session.get(url, params=params) #Use session to reuse connections and make everything run faster
            response = session.get(url)
            # print(f"This is the URL with parameters: {response.url}")
            # response = session.get(url)

        if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
            print(f"HTTP error for {vote_id}: ", response.status_code)
            print("Response text:", response.text)
            return None
        else:
            try:
                data = response.json()
            except ValueError:
                print("Error: Response is not valid JSON")
                print("Response text:", response.text)
                return None
        

        # print("Original_data", data)
        votes = data.get('value')
        next_link = data.get("odata.nextLink")
        # print("Next_link is ", next_link)
        # print("Value", votes)


        # votes = contained_data
        if not votes:
            # print("not votes????")
            break

        all_votes.extend(votes)
        if not next_link: #'https://oda.ft.dk/api/Afstemning(9700)/Stemme?$skip=100
            # print("No link found")
            break

        skip += 100
        # print("First loop done")

    return all_votes 

vote_id = 9700
data_from_voting_session = get_voting_session_with_votes(vote_id)


In [201]:
# voting_id = 9700#9799
# voting_id_2 = 9900#9798
# data_1 = get_voting_session_with_votes(voting_id)
# df_1 = get_data_from_voting_session(voting_id=voting_id, data_from_voting_session=data_1)
# data_2 = get_voting_session_with_votes(voting_id_2)
# df_2 = get_data_from_voting_session(voting_id=voting_id_2, data_from_voting_session=data_2)
# # pd.concat([df_1, df_2], axis = 1)
# df_1.compare(df_2)
# # df

In [ ]:
# voting_period = 69
#Save it as a period
def get_and_save_vote_id_from(df_voting_sessions, voting_period):
    vote_ids = get_voting_sessions_in_period(df_voting_sessions, voting_period)
    all_votes_full_period = []
    for vote_id in tqdm(vote_ids):
        votes_in_voting_session = get_voting_session_with_votes(vote_id)
        all_votes_full_period.extend(votes_in_voting_session)
    
    df = pd.DataFrame(all_votes_full_period)

    #Rename the columns to something usefull
    df.rename(columns={"id": "vote_id"
                       ,"typeid": "afstemning_typeid"
                       ,"afstemningid": "afstemning_id"
                       ,"opdateringsdato" : "afstemning_opdateringsdato"
                       ,"aktørid" : "afstemning_aktørid"
                       }
                       , inplace= True
                       )

    df.to_csv(f"./voting-data/df_votes_p{voting_period}.csv", index = False)
    return df

for voting_period in df_voting_sessions['Period'].unique():
    # print(f'Now working on period {period}')
    df = get_and_save_vote_id_from(df_voting_sessions, voting_period)

df.head(5)

Found 1727 votes in period 68


  7%|▋         | 121/1727 [00:07<01:32, 17.29it/s]

In [220]:
df = pd.DataFrame()
for period in df_votings['Period'].unique():
    with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/voting-data/df_votes_p{period}.csv") as response:
        df_votes_from_period = pd.read_csv(response)
        df = pd.concat([df, df_votes_from_period])

df.to_csv(f"./voting-data/df_votes_all_periods.csv", index = False)

In [221]:
df

,metadata_vote_id,vote_type,voting_id,aktør_id,metadata_opdateringsdato
0,1,1,1,158,2014-09-09T09:05:59.653
1,2,1,1,71,2014-09-09T09:05:59.653
2,3,1,1,50,2014-09-09T09:05:59.653
3,4,1,1,49,2014-09-09T09:05:59.653
4,5,1,1,183,2014-09-09T09:05:59.653
...,...,...,...,...,...
233395,2321106,1,10380,80,2025-10-24T09:45:23.913
233396,2321107,3,10380,18696,2025-10-24T09:45:23.913
233397,2321108,3,10380,18701,2025-10-24T09:45:23.913
233398,2321109,1,10380,18716,2025-10-24T09:45:23.913


In [222]:
periods = [65, 66, 67, 68, 69, 70, 71]
for period in periods:
    with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/voting-data/df_votes_p{period}.csv") as response:
        df = pd.read_csv(response)
        n_unique = df["aktør_id"].nunique()
        print(f"Period {period} has {n_unique} unique politicians")


Period 65 has 189 unique politicians
Period 66 has 216 unique politicians
Period 67 has 241 unique politicians
Period 68 has 228 unique politicians
Period 69 has 237 unique politicians
Period 70 has 219 unique politicians
Period 71 has 235 unique politicians


In [223]:
# period = 71
# with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/voting-data/df_votes_p{period}.csv") as response:
#     df = pd.read_csv(response)

# representatives = df[["politician", "party"]].drop_duplicates() #126 unique in 

# party_counts = (representatives.groupby(
#     by = ["party"]
# )
#     .size()
#     .rename("Member")
# )

# party_counts.head(20)

# # Alternativet mangler 5
# # Socialdemokratiet mangler 10
# # LA mangler 10 el. 11

# representatives[representatives["party"] == "Liberal Alliance"]
# #LA har Ole Birk Olesen, Carsten Bach, Henrik Dahl og Alex Vanopslagh
# #Mangler e.g. Alexandar Ryle